# 01 – CSV Pipeline Demo

This notebook walks through the end-to-end CSV processing pipeline:

1. Load the sample CSV from `data/raw/`
2. Inspect raw data
3. Run cleaning steps (normalize columns, handle missing values)
4. Run feature engineering
5. Compare before/after

> **Prerequisites**: `pip install -r requirements.txt && pip install -e .`

In [ ]:
import sys
import os

# Make sure the project root is on the path when running from notebooks/
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))

import pandas as pd

from src.processing.io import read_csv, write_csv, print_summary
from src.processing.cleaning import normalize_columns, strip_whitespace, drop_high_missing_columns, fill_missing_numeric, fill_missing_categorical, clean
from src.processing.features import add_row_id, add_missing_flag, normalize_numeric, encode_categorical

print('Imports OK')

## 1. Load Raw Data

In [ ]:
df_raw = read_csv('../data/raw/sample.csv')
print(f'Shape: {df_raw.shape}')
df_raw.head()

## 2. Raw Summary

In [ ]:
print_summary(df_raw)

## 3. Step-by-step Cleaning

In [ ]:
# 3a. Normalize column names
df1 = normalize_columns(df_raw)
print('After normalize_columns:')
print(df1.columns.tolist())

In [ ]:
# 3b. Strip whitespace from strings
df2 = strip_whitespace(df1)
print('After strip_whitespace:')
df2.head()

In [ ]:
# 3c. Drop columns with >= 50% missing
df3 = drop_high_missing_columns(df2, threshold=0.5)
print(f'Columns after drop: {df3.columns.tolist()}')

In [ ]:
# 3d. Fill missing numeric values
df4 = fill_missing_numeric(df3, strategy='median')
df4.isnull().sum()

In [ ]:
# 3e. Fill missing categorical
df5 = fill_missing_categorical(df4, fill='unknown')
df5.isnull().sum()

## 4. One-shot Clean + Feature Engineering

In [ ]:
df_clean = clean(df_raw)
df_out = add_row_id(df_clean)
print(f'Final shape: {df_out.shape}')
df_out.head()

## 5. Before / After Comparison

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    'Metric': ['Rows', 'Columns', 'Missing cells'],
    'Before': [len(df_raw), len(df_raw.columns), df_raw.isnull().sum().sum()],
    'After':  [len(df_out), len(df_out.columns), df_out.isnull().sum().sum()],
})
comparison

## 6. Save Processed CSV

In [ ]:
write_csv(df_out, '../data/processed/sample_clean.csv')
print('Done!')